# 02. Layer 2 — Runner / Session / Memory / Artifact

Layer 1 教你「Agent 是什麼」，Layer 2 教你「Agent 怎麼跑、跑完去哪」。四個元件分工：

| 元件 | 比喻 | 生命週期 |
|------|------|---------|
| **Runner** | Agent 的引擎 | 一次對話呼叫多次 |
| **Session** | 一場對話 | 一段使用者會話內 |
| **Memory** | 跨對話的長期記憶 | 跨 session 持久 |
| **Artifact** | Agent 系統的檔案系統 | 可選 session / user 範圍 |

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import get_model, final_text

## 1. Session — 同一場對話的狀態

Session 不只是「對話歷史」，它還有一個 `state` dict，可以讓 tool 寫入、後續輪次讀回。下面用 `output_key` 機制把 agent 回覆**自動寫入** session state。

In [2]:
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

name_agent = LlmAgent(
    name="name_agent",
    model=get_model(),
    instruction="使用者會告訴你他叫什麼名字。請只回覆對方的名字（一個詞），不要其他文字。",
    output_key="user_name",  # 自動把這次回覆寫到 session.state['user_name']
)

greeter_agent = LlmAgent(
    name="greeter",
    model=get_model(),
    instruction=(
        "用一句繁體中文歡迎使用者。如果 session state 裡有 user_name，"
        "就在歡迎詞裡叫他名字。\n\n"
        "使用者名字（從 state 注入）：{user_name?}"  # ? 表示 optional
    ),
)

APP, USER, SID = "layer2", "sean", "chat-1"
session_service = InMemorySessionService()
await session_service.create_session(app_name=APP, user_id=USER, session_id=SID)

Session(id='chat-1', app_name='layer2', user_id='sean', state={}, events=[], last_update_time=1777379340.4825292)

In [3]:
# Turn 1: 收名字，寫進 state
runner = Runner(agent=name_agent, app_name=APP, session_service=session_service)
msg = types.Content(role="user", parts=[types.Part(text="我叫 Sean")])
async for ev in runner.run_async(user_id=USER, session_id=SID, new_message=msg):
    if ev.is_final_response():
        print(f"name_agent 抓到的名字：{final_text(ev)}")

# Turn 2: 切換 agent，讀 state 出來打招呼
runner.agent = greeter_agent
msg = types.Content(role="user", parts=[types.Part(text="幫我介紹一下今天")])
async for ev in runner.run_async(user_id=USER, session_id=SID, new_message=msg):
    if ev.is_final_response():
        print(f"greeter 回覆：{final_text(ev)}")

# 看一下 session state
session = await session_service.get_session(app_name=APP, user_id=USER, session_id=SID)
print(f"\n📦 session.state = {dict(session.state)}")

Event from an unknown agent: name_agent, event id: ca5f50bb-f639-4620-a7a6-7c1c7b81670c


name_agent 抓到的名字：Sean


greeter 回覆：您好，Sean！  
今天是西元 2026 年 4 月 28 日，星期三。目前正值春季中段，在北半球許多地區氣溫回暖、花草開始綻放；而南半球則進入秋季，葉子逐漸轉紅。距離今年結束還剩下約247天，接下來的周末將於本週六和日到來，是個適合外出踏青或安排小旅行的好時機。如需更詳細的新聞摘要或特定領域資訊，也請隨時告訴我！

📦 session.state = {'user_name': 'Sean'}


## 2. Memory — 跨 Session 的長期記憶

Session 裡的東西，session 結束就消失。**Memory** 是跨 session 的儲存：你可以把整個 session 的對話「歸檔」到 memory，下次新 session 開了，agent 用 `load_memory` 工具去搜尋過往記憶。

**Session vs Memory 的差別**：
- Session = RAM（短期、結構化、結束就清）
- Memory = 硬碟（長期、可搜尋、跨對話）

In [4]:
from google.adk.memory import InMemoryMemoryService
from google.adk.tools import load_memory

memory_service = InMemoryMemoryService()

# 把第一個 session 歸檔到 memory
session1 = await session_service.get_session(app_name=APP, user_id=USER, session_id=SID)
await memory_service.add_session_to_memory(session1)
print("已把 session 1 寫入 memory")

已把 session 1 寫入 memory


In [5]:
# 開一個新 session，用 load_memory 工具回憶
memory_agent = LlmAgent(
    name="memory_agent",
    model=get_model(),
    instruction=(
        "你能存取使用者過去的對話。"
        "當使用者問跟過往有關的問題時，**先呼叫 load_memory 工具**搜尋相關片段，再用一句繁體中文回答。"
    ),
    tools=[load_memory],
)

SID2 = "chat-2"
await session_service.create_session(app_name=APP, user_id=USER, session_id=SID2)
runner2 = Runner(
    agent=memory_agent,
    app_name=APP,
    session_service=session_service,
    memory_service=memory_service,  # ← 一定要把 memory_service 傳進來
)

msg = types.Content(role="user", parts=[types.Part(text="你還記得我叫什麼名字嗎？")])
async for ev in runner2.run_async(user_id=USER, session_id=SID2, new_message=msg):
    if ev.get_function_calls():
        for c in ev.get_function_calls():
            print(f"  🔧 {c.name}({c.args})")
    if ev.is_final_response():
        print(f"\n=== 回覆 ===\n{final_text(ev)}")

  🔧 load_memory({'query': "User's name"})



=== 回覆 ===
您的名字是 Sean。


## 3. Artifact — Agent 系統的檔案系統

Memory 是文字記憶，**Artifact 是檔案儲存** — 適合存報告、生成的 PDF、上傳的圖片等。Tool 透過 `tool_context.save_artifact()` / `load_artifact()` 操作。

**Namespacing**：`filename` 開頭加 `user:` 前綴可以跨 session 共用（user-scoped），不加前綴是這個 session 限定。

In [6]:
from google.adk.artifacts import InMemoryArtifactService
from google.adk.tools.tool_context import ToolContext

async def save_summary_report(topic: str, content: str, tool_context: ToolContext) -> dict:
    """Save a markdown summary report as a user-scoped artifact.

    Args:
        topic: Short topic / title.
        content: Markdown body of the report.
    """
    md = f"# {topic}\n\n{content}\n"
    version = await tool_context.save_artifact(
        filename="user:summary.md",  # user: 前綴 → 跨 session
        artifact=types.Part(text=md),
    )
    return {"saved": True, "version": version, "length": len(md)}

async def read_summary_report(tool_context: ToolContext) -> dict:
    """Read the latest summary report from artifacts."""
    part = await tool_context.load_artifact(filename="user:summary.md")
    if part is None:
        return {"found": False}
    return {"found": True, "content": part.text}

from google.adk.tools import FunctionTool
writer_agent = LlmAgent(
    name="writer_agent",
    model=get_model(),
    instruction=(
        "使用者會請你寫一份簡短摘要。**呼叫 save_summary_report 工具**儲存，"
        "再用一句話告訴使用者已存好。"
    ),
    tools=[FunctionTool(func=save_summary_report)],
)

reader_agent = LlmAgent(
    name="reader_agent",
    model=get_model(),
    instruction="當使用者要看上次的摘要時，**呼叫 read_summary_report 工具**取出，再原樣回覆內容。",
    tools=[FunctionTool(func=read_summary_report)],
)

In [7]:
artifact_service = InMemoryArtifactService()

# Session A: 寫
SID_A = "art-a"
await session_service.create_session(app_name=APP, user_id=USER, session_id=SID_A)
runner_w = Runner(
    agent=writer_agent,
    app_name=APP,
    session_service=session_service,
    artifact_service=artifact_service,
)
msg = types.Content(role="user", parts=[types.Part(
    text="幫我寫一份『ADK 是什麼』的兩句摘要，存起來。"
)])
async for ev in runner_w.run_async(user_id=USER, session_id=SID_A, new_message=msg):
    if ev.get_function_responses():
        for r in ev.get_function_responses():
            print(f"  📦 save 結果：{r.response}")
    if ev.is_final_response():
        print(f"  ✅ {final_text(ev)}")

  📦 save 結果：{'saved': True, 'version': 0, 'length': 138}


  ✅ 已為您將「ADK 是什麼」的二行摘要保存完畢。


In [8]:
# Session B（不同 session！）：讀
SID_B = "art-b"
await session_service.create_session(app_name=APP, user_id=USER, session_id=SID_B)
runner_r = Runner(
    agent=reader_agent,
    app_name=APP,
    session_service=session_service,
    artifact_service=artifact_service,  # 共用同一個 artifact service
)
msg = types.Content(role="user", parts=[types.Part(text="我上次寫的摘要呢？")])
async for ev in runner_r.run_async(user_id=USER, session_id=SID_B, new_message=msg):
    if ev.is_final_response():
        print(f"=== 跨 session 取回的摘要 ===\n{final_text(ev)}")

=== 跨 session 取回的摘要 ===
以下是您上次撰寫的摘要：

```
# ADK 是什麼

ADK（Android Development Kit）是 Google 為 Android 平台提供的一組開發工具與函式庫，包含了 SDK、模擬器以及相關說明文件。
它讓開發者能在電腦上編譯、測試並除錯 Android 手機和平板等裝置上的應用程式。
```


## 結論

現在你看得懂 Agent **跑起來**會發生什麼事：

- **Runner** 是執行引擎，串起 session / memory / artifact 三個服務
- **Session** = 短期記憶，state 跟對話歷史在這裡
- **Memory** = 長期記憶，存的是「過去的對話」可以被 agent 主動搜尋
- **Artifact** = 檔案儲存，存的是「結構化檔案」可以跨 session

**搭配建議**：簡單 demo 用 `InMemory*` 系列服務（不持久），上 production 換成 `DatabaseSessionService`、`VertexAiMemoryBankService`、`GcsArtifactService`。

下一站：`03_layer3_orchestration.ipynb` — 多 Agent 流程型協作。